In [8]:
#load datd 
import pandas as pd
import os

# Load product metadata
df = pd.read_csv('data/styles.csv', on_bad_lines='skip')
df = df.dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)

print("Sample rows:")
print(df[['id', 'productDisplayName', 'articleType']].head())

print("Number of images available:", len(os.listdir('data/images')))


Sample rows:
      id                             productDisplayName  articleType
0  15970               Turtle Check Men Navy Blue Shirt       Shirts
1  39386             Peter England Men Party Blue Jeans        Jeans
2  59263                       Titan Women Silver Watch      Watches
3  21379  Manchester United Men Solid Black Track Pants  Track Pants
4  53759                          Puma Men Grey T-shirt      Tshirts
Number of images available: 44441


In [9]:
import cv2
import numpy as np
from tqdm import tqdm

image_features = []
valid_ids = []

for img_id in tqdm(df['id'].values):
    path = f"data/images/{img_id}.jpg"
    try:
        img = cv2.imread(path)
        if img is not None:
            img = cv2.resize(img, (32, 32))
            image_features.append(img / 255.0)
            valid_ids.append(img_id)
    except:
        continue

image_features = np.array(image_features)
print("Processed images:", image_features.shape)


100%|███████| 44417/44417 [00:53<00:00, 824.98it/s]


Processed images: (44412, 32, 32, 3)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Filter DataFrame to match only valid image IDs
df_filtered = df[df['id'].isin(valid_ids)].dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)

# Extract cleaned text and labels
texts = df_filtered['productDisplayName'].astype(str).values
labels = df_filtered['articleType'].astype(str).values

# TF-IDF vectorization (NLP)
vectorizer = TfidfVectorizer(max_features=500)
text_features = vectorizer.fit_transform(texts).toarray()

# Align image features with filtered text rows
image_features_flat = image_features[:len(df_filtered)].reshape(len(df_filtered), -1)

# Combine text and image features
import numpy as np
combined_features = np.hstack([text_features, image_features_flat])

print("Text features:", text_features.shape)
print("Image features:", image_features_flat.shape)
print("Combined features:", combined_features.shape)


Text features: (44412, 500)
Image features: (44412, 3072)
Combined features: (44412, 3572)


In [11]:
from sklearn.metrics import classification_report
import numpy as np

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))

# Fix for the label mismatch error
labels_used = np.unique(y_test)

# Classification report for only used labels
print("\n📊 Classification Report:\n", classification_report(
    y_test, y_pred,
    labels=labels_used,
    target_names=encoder.inverse_transform(labels_used)
))


NameError: name 'accuracy_score' is not defined

In [12]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))

# Fix for the label mismatch error
labels_used = np.unique(y_test)

# Classification report for only used labels
print("\n📊 Classification Report:\n", classification_report(
    y_test, y_pred,
    labels=labels_used,
    target_names=encoder.inverse_transform(labels_used)
))


NameError: name 'y_test' is not defined

In [13]:
# 1) Imports
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import accuracy_score, classification_report

# 2) Re-create train/test split (using your already-computed combined_features & labels)
encoder = LabelEncoder()
y_enc = encoder.fit_transform(labels)              # 'labels' must already be in memory
X_train, X_test, y_train, y_test = train_test_split(
    combined_features, y_enc, test_size=0.2, random_state=42
)

# 3) Train the model (or reload if you saved it)
model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X_train, y_train)

# 4) Predict
y_pred = model.predict(X_test)

# 5) Evaluate
print("✅ Accuracy:", accuracy_score(y_test, y_pred))

# Only include the classes present in the test split
labels_used = np.unique(y_test)
print("\n📊 Classification Report:\n",
      classification_report(
          y_test, y_pred,
          labels=labels_used,
          target_names=encoder.inverse_transform(labels_used)
      )
)


✅ Accuracy: 0.7959022852639874

📊 Classification Report:
                         precision    recall  f1-score   support

    Accessory Gift Set       0.94      1.00      0.97        17
            Baby Dolls       0.00      0.00      0.00         7
             Backpacks       0.79      0.93      0.85       162
                Bangle       0.55      0.85      0.67        13
           Basketballs       0.50      0.25      0.33         4
             Bath Robe       0.00      0.00      0.00         2
                 Belts       0.89      0.99      0.93       160
               Blazers       0.00      0.00      0.00         2
           Body Lotion       0.00      0.00      0.00         1
               Booties       1.00      0.25      0.40         4
                Boxers       1.00      0.23      0.38        13
                   Bra       0.86      0.91      0.88        91
              Bracelet       0.80      0.53      0.64        15
                Briefs       0.88      0.97  

C:\Users\user\automated-product-classification\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\user\automated-product-classification\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\user\automated-product-classification\venv\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f

In [14]:
import joblib

# Save classifier
joblib.dump(model, 'product_classifier_rf.pkl')

# Save TF-IDF vectorizer
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("✅ Saved model and vectorizer!")


✅ Saved model and vectorizer!


In [5]:
import pandas as pd

# 1. Load dataset
df = pd.read_csv("data/styles.csv", on_bad_lines='skip')
df = df.dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)

# 2. Load valid image IDs (from earlier image processing)
# If valid_ids is also gone, re-create it like this:
import os
valid_ids = [int(filename.split('.')[0]) for filename in os.listdir('data/images') if filename.endswith('.jpg')]

# 3. Filter only rows with valid images
df_filtered = df[df['id'].isin(valid_ids)].dropna(subset=['productDisplayName', 'articleType']).reset_index(drop=True)

# 4. Get the labels
labels = df_filtered['articleType'].values

# 5. Create and save LabelEncoder
from sklearn.preprocessing import LabelEncoder
import joblib

encoder = LabelEncoder()
encoder.fit(labels)
joblib.dump(encoder, 'label_encoder.pkl')

print("✅ Everything ready! LabelEncoder saved as 'label_encoder.pkl'")


✅ Everything ready! LabelEncoder saved as 'label_encoder.pkl'
